In [1]:
print("hello world")

hello world


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter

# Identify project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = next(RAW_DIR.glob("accepted_*.csv*"))

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Project: d:\M_Tech_IIITB\Projects\Data-Analysis\credit-risk-analytics
Dataset: d:\M_Tech_IIITB\Projects\Data-Analysis\credit-risk-analytics\data\raw\accepted_2007_to_2018Q4.csv.gz
Dataset exists: True


In [3]:
TARGET_STATUSES = [
    "Fully Paid",
    "Charged Off",
    "Default"
]

status_counts = Counter()
application_counts = Counter()
vintage_counts = Counter()

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "loan_status",
        "issue_d",
        "term",
        "application_type"
    ],
    chunksize=200_000
):

    chunk = chunk[
        chunk["loan_status"].isin(TARGET_STATUSES)
    ].copy()

    issue_date = pd.to_datetime(
        chunk["issue_d"],
        format="%b-%Y",
        errors="coerce"
    )

    issue_period = issue_date.dt.to_period("M")

    term_months = pd.to_numeric(
        chunk["term"].astype("string").str.extract(r"(\d+)")[0],
        errors="coerce"
    )

    mature_mask = (
        (
            (term_months == 36) &
            (issue_period <= pd.Period("2015-12", freq="M"))
        )
        |
        (
            (term_months == 60) &
            (issue_period <= pd.Period("2013-12", freq="M"))
        )
    )

    chunk = chunk[mature_mask].copy()

    status_counts.update(
        chunk["loan_status"].value_counts().to_dict()
    )

    application_counts.update(
        chunk["application_type"]
        .fillna("Missing")
        .value_counts()
        .to_dict()
    )

    chunk["issue_year"] = issue_date[mature_mask].dt.year
    vintage_counts.update(
        chunk.groupby(["issue_year", "term"]).size().to_dict()
    )

print("Mature cohort status distribution:")
display(pd.Series(status_counts, name="loan_count"))

print("\nApplication type distribution:")
display(pd.Series(application_counts, name="loan_count"))

print("\nMature cohort by vintage:")
display(
    pd.DataFrame(
        [
            {"year": k[0], "term": k[1], "count": v}
            for k, v in vintage_counts.items()
        ]
    ).sort_values(["year", "term"])
)

Mature cohort status distribution:


Fully Paid     573768
Charged Off     99785
Name: loan_count, dtype: int64


Application type distribution:


Individual    673314
Joint App        239
Name: loan_count, dtype: int64


Mature cohort by vintage:


,year,term,count
2,2007,36 months,251
3,2008,36 months,1562
4,2009,36 months,4716
5,2010,36 months,8466
6,2010,60 months,3070
7,2011,36 months,14101
8,2011,60 months,7620
11,2012,36 months,43470
12,2012,60 months,9897
9,2013,36 months,100422


In [4]:
# defining target and features for the model

CANDIDATE_FEATURES = [
    "loan_amnt",
    "term",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "fico_range_low"
]

TARGET_STATUSES = [
    "Fully Paid",
    "Charged Off",
    "Default"
]

print("Candidate predictors:", len(CANDIDATE_FEATURES))

Candidate predictors: 16


In [5]:
usecols = CANDIDATE_FEATURES + [
    "loan_status",
    "issue_d",
    "application_type"
]

chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    chunksize=200_000
):
    # Retain resolved outcomes
    chunk = chunk[
        chunk["loan_status"].isin(TARGET_STATUSES)
    ].copy()

    # Parse origination date
    chunk["issue_date"] = pd.to_datetime(
        chunk["issue_d"],
        format="%b-%Y",
        errors="coerce"
    )

    # Extract loan duration
    term_months = pd.to_numeric(
        chunk["term"].astype("string").str.extract(r"(\d+)")[0],
        errors="coerce"
    )

    # Apply maturity criteria
    mature_mask = (
        (
            (term_months == 36) &
            (chunk["issue_date"] <= pd.Timestamp("2015-12-01"))
        )
        |
        (
            (term_months == 60) &
            (chunk["issue_date"] <= pd.Timestamp("2013-12-01"))
        )
    )

    chunk = chunk[mature_mask].copy()

    # Initial project: individual applications only
    chunk = chunk[
        chunk["application_type"] == "Individual"
    ].copy()

    # Binary target
    chunk["default_flag"] = (
        chunk["loan_status"].isin(
            ["Charged Off", "Default"]
        )
    ).astype(int)

    # Preserve origination year for temporal splitting
    chunk["issue_year"] = chunk["issue_date"].dt.year

    # Deterministic data-quality corrections
    chunk.loc[
        chunk["annual_inc"] <= 0, "annual_inc"
    ] = np.nan

    chunk.loc[
        chunk["dti"] == 999, "dti"
    ] = np.nan

    chunks.append(chunk)

# Combine eligible observations
model_df = pd.concat(chunks, ignore_index=True)

# Remove administrative columns that aren't predictors
model_df.drop(
    columns=["loan_status", "application_type", "issue_d"],
    inplace=True
)

print("Final modelling dataset:", model_df.shape)

print("\nTarget distribution:")
display(model_df["default_flag"].value_counts())

print("\nDefault rate:")
print(f"{model_df['default_flag'].mean():.2%}")

print("\nOrigination year distribution:")
display(model_df["issue_year"].value_counts().sort_index())

Final modelling dataset: (673314, 19)

Target distribution:


C:\Users\mihir\AppData\Local\Temp\ipykernel_50044\3239612570.py:77: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  model_df.drop(


default_flag
0    573572
1     99742
Name: count, dtype: int64


Default rate:
14.81%

Origination year distribution:


issue_year
2007       251
2008      1562
2009      4716
2010     11536
2011     21721
2012     53367
2013    134804
2014    162570
2015    282787
Name: count, dtype: int64

In [6]:
OUTPUT_PATH = (
    PROCESSED_DIR / "credit_risk_modeling_cohort.csv.gz"
)

model_df.to_csv(
    OUTPUT_PATH,
    index=False,
    compression="gzip"
)

print("Saved successfully:", OUTPUT_PATH)

Saved successfully: d:\M_Tech_IIITB\Projects\Data-Analysis\credit-risk-analytics\data\processed\credit_risk_modeling_cohort.csv.gz


Cohort Validation and Temporal Split Diagnostics

In [7]:
# Normalize term labels
model_df["term"] = model_df["term"].astype("string").str.strip()

# 1. Validate dataset
print("Dataset shape:", model_df.shape)
print("\nTerm distribution:")
print(model_df["term"].value_counts(dropna=False))

print("\nMissing values (%):")
print((model_df.isna().mean() * 100).round(2).sort_values(ascending=False))

# 2. Loan counts by origination year and term
term_by_year = pd.crosstab(
    model_df["issue_year"],
    model_df["term"]
)

print("\nLoan counts by year and term:")
display(term_by_year)

# 3. Default rates by origination year
vintage_summary = (
    model_df.groupby("issue_year")
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

vintage_summary["default_rate"] *= 100

print("\nDefault rates by origination year:")
display(vintage_summary.round(2))

# 4. Default rates by year and loan term
term_summary = (
    model_df.groupby(["issue_year", "term"])
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

term_summary["default_rate"] *= 100

print("\nDefault rates by year and term:")
display(term_summary.round(2))

Dataset shape: (673314, 19)

Term distribution:
term
36 months    618345
60 months     54969
Name: count, dtype: Int64

Missing values (%):
emp_length             5.57
revol_util             0.06
loan_amnt              0.00
term                   0.00
home_ownership         0.00
verification_status    0.00
annual_inc             0.00
dti                    0.00
delinq_2yrs            0.00
fico_range_low         0.00
purpose                0.00
inq_last_6mths         0.00
open_acc               0.00
pub_rec                0.00
revol_bal              0.00
total_acc              0.00
issue_date             0.00
default_flag           0.00
issue_year             0.00
dtype: float64

Loan counts by year and term:


term,36 months,60 months
issue_year,,
2007,251,0
2008,1562,0
2009,4716,0
2010,8466,3070
2011,14101,7620
2012,43470,9897
2013,100422,34382
2014,162570,0
2015,282787,0



Default rates by origination year:


,loans,defaults,default_rate
issue_year,,,
2007,251,45,17.93
2008,1562,247,15.81
2009,4716,594,12.60
2010,11536,1487,12.89
2011,21721,3297,15.18
2012,53367,8644,16.20
2013,134804,21024,15.60
2014,162570,22315,13.73
2015,282787,42089,14.88



Default rates by year and term:


loans  defaults  default_rate
issue_year term                                     
2007       36 months     251        45         17.93
2008       36 months    1562       247         15.81
2009       36 months    4716       594         12.60
2010       36 months    8466       842          9.95
           60 months    3070       645         21.01
2011       36 months   14101      1499         10.63
           60 months    7620      1798         23.60
2012       36 months   43470      5903         13.58
           60 months    9897      2741         27.70
2013       36 months  100422     12378         12.33
           60 months   34382      8646         25.15
2014       36 months  162570     22315         13.73
2015       36 months  282787     42089         14.88

36-Month Cohort and Temporal Train-Validation-Test Split

In [8]:
# Keep only 36-month loans for the baseline model
model_36m = model_df[
    model_df["term"] == "36 months"
].copy()

print("36-month cohort shape:", model_36m.shape)

# Temporal split
train_df = model_36m[
    model_36m["issue_year"] <= 2013
].copy()

val_df = model_36m[
    model_36m["issue_year"] == 2014
].copy()

test_df = model_36m[
    model_36m["issue_year"] == 2015
].copy()

# Verify split sizes and target distributions
for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print("Rows:", len(subset))
    print("Defaults:", subset["default_flag"].sum())
    print(
        "Default rate:",
        f"{subset['default_flag'].mean():.2%}"
    )

# Verify chronological separation
assert train_df["issue_year"].max() < val_df["issue_year"].min()
assert val_df["issue_year"].max() < test_df["issue_year"].min()

print("\nTemporal split verified successfully.")

36-month cohort shape: (618345, 19)

Train
Rows: 172988
Defaults: 21508
Default rate: 12.43%

Validation
Rows: 162570
Defaults: 22315
Default rate: 13.73%

Test
Rows: 282787
Defaults: 42089
Default rate: 14.88%

Temporal split verified successfully.


In [9]:
# Exclude term because it is constant in the 36-month cohort
MODEL_FEATURES = [
    col for col in CANDIDATE_FEATURES
    if col != "term"
]

TARGET = "default_flag"

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[MODEL_FEATURES].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df[TARGET].copy()

print("Number of model features:", len(MODEL_FEATURES))
print("Training feature matrix:", X_train.shape)
print("Validation feature matrix:", X_val.shape)
print("Test feature matrix:", X_test.shape)

Number of model features: 15
Training feature matrix: (172988, 15)
Validation feature matrix: (162570, 15)
Test feature matrix: (282787, 15)
